# LeetCode 1422 — Maximum Score After Splitting a String bằng Python

**Link đề:** https://leetcode.com/problems/maximum-score-after-splitting-a-string/
**Độ khó:** Easy · **Chủ đề:** String, Prefix Sum

```python
class Solution:
    def maxScore(self, s: str) -> int:
        ...
```

## Mục tiêu học
1. Liệt kê mọi **điểm cắt** hợp lệ của chuỗi bằng slicing `s[:i]`, `s[i:]` — và hiểu vì sao `i` chạy từ `1` tới `n − 1`.
2. Viết baseline đếm lại cho từng điểm cắt (`O(n²)`).
3. Dùng **prefix sum** để mỗi điểm cắt chỉ tốn `O(1)` → `O(n)`.
4. Biến đổi công thức để chỉ cần **một lượt duyệt** (`ones_total + max(zeros − ones)`).
5. Trace, chứng minh, Time/Space, lỗi hay gặp (cắt tại `i = 0` hoặc `i = n`), bài tập.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài

Cho chuỗi `s` chỉ gồm `'0'` và `'1'`. Cắt `s` thành **hai chuỗi con khác rỗng**: `left` và `right`.

**Điểm** của một cách cắt = (số `'0'` trong `left`) + (số `'1'` trong `right`). Trả về **điểm lớn nhất**.

| `s` | Output | Cách cắt tốt nhất |
|---|---:|---|
| `"011101"` | `5` | `"0"` | `"11101"` → 1 + 4 |
| `"00111"` | `5` | `"00"` | `"111"` → 2 + 3 |
| `"1111"` | `3` | `"1"` | `"111"` → 0 + 3 (left buộc phải khác rỗng) |

**Constraints:** `2 <= s.length <= 500`, `s` chỉ gồm `'0'` và `'1'`.

In [1]:
examples = [("011101", 5), ("00111", 5), ("1111", 3)]
for s, expected in examples:
    print(f"s = {s!r} -> kỳ vọng {expected}")

s = '011101' -> kỳ vọng 5
s = '00111' -> kỳ vọng 5
s = '1111' -> kỳ vọng 3


## 2. Nền tảng Python cần biết

### 2.1. Cắt chuỗi `s[:i]` và `s[i:]`

- `s[:i]` = `i` ký tự đầu (chỉ số `0..i−1`).
- `s[i:]` = phần còn lại (chỉ số `i..n−1`).
- Hai phần **khác rỗng** ⇔ `1 <= i <= n − 1`.

In [2]:
s = "011101"
for i in range(len(s) + 1):
    note = "" if 1 <= i <= len(s) - 1 else "  <- KHÔNG hợp lệ (có phần rỗng)"
    print(f"i={i}: left={s[:i]!r:<9} right={s[i:]!r:<9}{note}")

i=0: left=''        right='011101'   <- KHÔNG hợp lệ (có phần rỗng)
i=1: left='0'       right='11101'  
i=2: left='01'      right='1101'   
i=3: left='011'     right='101'    
i=4: left='0111'    right='01'     
i=5: left='01110'   right='1'      
i=6: left='011101'  right=''         <- KHÔNG hợp lệ (có phần rỗng)


### 2.2. `str.count` — đếm ký tự

In [3]:
print("'011101'.count('0') =", "011101".count("0"))
print("'011101'.count('1') =", "011101".count("1"))

'011101'.count('0') = 2
'011101'.count('1') = 4


### 2.3. Prefix sum (tổng tiền tố)

`ones[i]` = số `'1'` trong `s[:i]` (tức `i` ký tự đầu). Tạo mảng dài `n + 1` với `ones[0] = 0`:

`ones[i + 1] = ones[i] + (1 if s[i] == '1' else 0)`.

Sau đó:
- số `'1'` trong `left = s[:i]` là `ones[i]`;
- số `'0'` trong `left` là `i − ones[i]`;
- số `'1'` trong `right = s[i:]` là `ones[n] − ones[i]`.

In [4]:
s = "011101"
ones = [0] * (len(s) + 1)
for i, ch in enumerate(s):
    ones[i + 1] = ones[i] + (ch == "1")
print("s    =", list(s))
print("ones =", ones, "(ones[i] = số '1' trong s[:i])")

s    = ['0', '1', '1', '1', '0', '1']
ones = [0, 0, 1, 2, 3, 3, 4] (ones[i] = số '1' trong s[:i])


`ones[i] + (ch == "1")`: biểu thức so sánh cho `True`/`False`, cộng với số nguyên thì `True` là `1`, `False` là `0`.

### 2.4. `int(ch)` cho ký tự `'0'`/`'1'`

`int('1') = 1`, `int('0') = 0` — một cách khác để biến ký tự thành số khi đếm.

In [5]:
print([int(ch) for ch in "0110"], sum(int(ch) for ch in "0110"))

[0, 1, 1, 0] 2


## 3. Cách trực giác (baseline) — thử mọi điểm cắt, đếm lại

Với mỗi `i` từ `1` đến `n − 1`: `score = s[:i].count('0') + s[i:].count('1')`. Giữ max.

In [6]:
def max_score_bruteforce(s: str) -> int:
    best = 0
    for i in range(1, len(s)):
        score = s[:i].count("0") + s[i:].count("1")
        best = max(best, score)
    return best

for s, expected in examples:
    print(s, "->", max_score_bruteforce(s), "| kỳ vọng", expected)

011101 -> 5 | kỳ vọng 5
00111 -> 5 | kỳ vọng 5
1111 -> 3 | kỳ vọng 3


### 3.1. Nút thắt

Có `n − 1` điểm cắt; mỗi điểm cắt tạo 2 chuỗi con và đếm lại toàn bộ `n` ký tự ⇒ `O(n²)`. Với `n = 500` vẫn ổn (~250 000 phép), nhưng có sự **lặp việc**: khi dời điểm cắt sang phải 1 ô, chỉ **một ký tự** đổi phe — không cần đếm lại từ đầu.

In [7]:
s = "011101"
for i in range(1, len(s)):
    print(f"i={i}: ký tự chuyển từ right sang left khi đi từ i-1 tới i là s[{i - 1}]={s[i - 1]!r}")

i=1: ký tự chuyển từ right sang left khi đi từ i-1 tới i là s[0]='0'
i=2: ký tự chuyển từ right sang left khi đi từ i-1 tới i là s[1]='1'
i=3: ký tự chuyển từ right sang left khi đi từ i-1 tới i là s[2]='1'
i=4: ký tự chuyển từ right sang left khi đi từ i-1 tới i là s[3]='1'
i=5: ký tự chuyển từ right sang left khi đi từ i-1 tới i là s[4]='0'


## 4. Insight — mỗi bước dời điểm cắt chỉ thay đổi 1 ký tự

Khi điểm cắt dời từ `i` sang `i + 1`, ký tự `s[i]` chuyển từ `right` sang `left`:
- Nếu `s[i] == '0'`: `left` thêm một số 0 ⇒ điểm **+1**.
- Nếu `s[i] == '1'`: `right` mất một số 1 ⇒ điểm **−1**.

Vậy chỉ cần biết điểm tại một vị trí ban đầu, rồi cập nhật `±1`. Đây là tư duy **prefix sum** / **cửa sổ trượt điểm cắt**.

Bắt đầu tại `i = 0` (chưa hợp lệ, chỉ để tính): `left` rỗng, `right = s` ⇒ điểm = `s.count('1')`.

In [8]:
s = "011101"
score = s.count("1")          # điểm "ảo" tại i = 0
print(f"i=0 (ảo): score={score}")
for i in range(len(s) - 1):   # dời tới i+1 = 1..n-1
    score += 1 if s[i] == "0" else -1
    print(f"i={i + 1}: s[{i}]={s[i]!r} -> score={score} | kiểm tra: {s[:i + 1].count('0') + s[i + 1:].count('1')}")

i=0 (ảo): score=4
i=1: s[0]='0' -> score=5 | kiểm tra: 5
i=2: s[1]='1' -> score=4 | kiểm tra: 4
i=3: s[2]='1' -> score=3 | kiểm tra: 3
i=4: s[3]='1' -> score=2 | kiểm tra: 2
i=5: s[4]='0' -> score=3 | kiểm tra: 3


## 5. Lời giải chính — prefix sum, giảng từng đoạn

**Đoạn 1:** xây `ones` độ dài `n + 1`.

**Đoạn 2:** với `i = 1..n−1`: `zeros_left = i − ones[i]`, `ones_right = ones[n] − ones[i]`.

**Đoạn 3:** `best = max(best, zeros_left + ones_right)`.

In [9]:
def max_score_prefix(s: str) -> int:
    n = len(s)
    ones = [0] * (n + 1)                          # Đoạn 1
    for i, ch in enumerate(s):
        ones[i + 1] = ones[i] + (ch == "1")
    best = 0
    for i in range(1, n):                         # Đoạn 2
        zeros_left = i - ones[i]
        ones_right = ones[n] - ones[i]
        best = max(best, zeros_left + ones_right) # Đoạn 3
    return best

for s, expected in examples:
    print(s, "->", max_score_prefix(s), "| kỳ vọng", expected)

011101 -> 5 | kỳ vọng 5
00111 -> 5 | kỳ vọng 5
1111 -> 3 | kỳ vọng 3


## 6. Cách 3 — một lượt, không mảng phụ: cập nhật `±1`

Từ insight mục 4: `score` bắt đầu bằng `s.count('1')`, rồi với `i = 0..n−2` cộng `+1` nếu `s[i] == '0'`, `−1` nếu `'1'`. Lấy max sau mỗi bước.

(Ta dừng ở `n − 2` vì điểm cắt cuối cùng hợp lệ là `i + 1 = n − 1`.)

In [10]:
def max_score_delta(s: str) -> int:
    score = s.count("1")
    best = 0
    for i in range(len(s) - 1):
        score += 1 if s[i] == "0" else -1
        best = max(best, score)
    return best

for s, expected in examples:
    print(s, "->", max_score_delta(s), "| kỳ vọng", expected)

011101 -> 5 | kỳ vọng 5
00111 -> 5 | kỳ vọng 5
1111 -> 3 | kỳ vọng 3


## 7. Cách 4 — tách hằng số ra khỏi max (công thức đẹp)

Tại điểm cắt `i`: `score = zeros_left + (ones_total − ones_left) = ones_total + (zeros_left − ones_left)`.

`ones_total` **không phụ thuộc** `i` ⇒ chỉ cần tìm `max (zeros_left − ones_left)` qua các `i = 1..n−1`. Một lượt duyệt, đếm `zeros`, `ones` của phần trái; **không cần** `s.count` trước (có thể đếm `ones_total` trong cùng lượt bằng cách cộng thêm ký tự cuối).

In [11]:
def max_score_one_pass(s: str) -> int:
    zeros = ones = 0
    best_diff = float("-inf")
    for i in range(len(s) - 1):          # left = s[:i+1], i+1 = 1..n-1
        if s[i] == "0":
            zeros += 1
        else:
            ones += 1
        best_diff = max(best_diff, zeros - ones)
    ones += s[-1] == "1"                 # cộng ký tự cuối -> ones = tổng số '1'
    return ones + best_diff

for s, expected in examples:
    print(s, "->", max_score_one_pass(s), "| kỳ vọng", expected)

011101 -> 5 | kỳ vọng 5
00111 -> 5 | kỳ vọng 5
1111 -> 3 | kỳ vọng 3


`float("-inf")` là **âm vô cùng** — mọi số đều lớn hơn nó, dùng làm giá trị khởi tạo cho `max`. `s[-1]` là ký tự **cuối** chuỗi.

## 8. TRACE từng bước

Trace cách 4 với `s = "011101"` (`ones_total = 4`):

| i | s[i] | left | zeros | ones | zeros − ones | best_diff |
|---:|---|---|---:|---:|---:|---:|
| 0 | 0 | `0` | 1 | 0 | 1 | **1** |
| 1 | 1 | `01` | 1 | 1 | 0 | 1 |
| 2 | 1 | `011` | 1 | 2 | −1 | 1 |
| 3 | 1 | `0111` | 1 | 3 | −2 | 1 |
| 4 | 0 | `01110` | 2 | 3 | −1 | 1 |

Ký tự cuối `'1'` ⇒ `ones = 3 + 1 = 4` ⇒ kết quả `4 + 1 = 5` ✔.

In [12]:
def trace_max_score(s: str) -> int:
    zeros = ones = 0
    best_diff = float("-inf")
    print(f"s = {s!r}")
    for i in range(len(s) - 1):
        if s[i] == "0":
            zeros += 1
        else:
            ones += 1
        old = best_diff
        best_diff = max(best_diff, zeros - ones)
        print(f"  i={i} s[i]={s[i]!r} left={s[:i + 1]!r:<8} zeros={zeros} ones={ones} "
              f"diff={zeros - ones:>2} best_diff: {old} -> {best_diff}")
    total_ones = ones + (s[-1] == "1")
    print(f"  ones_total = {total_ones} -> đáp án = {total_ones} + {best_diff} = {total_ones + best_diff}")
    return total_ones + best_diff

trace_max_score("011101")
print()
trace_max_score("1111")

s = '011101'
  i=0 s[i]='0' left='0'      zeros=1 ones=0 diff= 1 best_diff: -inf -> 1
  i=1 s[i]='1' left='01'     zeros=1 ones=1 diff= 0 best_diff: 1 -> 1
  i=2 s[i]='1' left='011'    zeros=1 ones=2 diff=-1 best_diff: 1 -> 1
  i=3 s[i]='1' left='0111'   zeros=1 ones=3 diff=-2 best_diff: 1 -> 1
  i=4 s[i]='0' left='01110'  zeros=2 ones=3 diff=-1 best_diff: 1 -> 1
  ones_total = 4 -> đáp án = 4 + 1 = 5

s = '1111'
  i=0 s[i]='1' left='1'      zeros=0 ones=1 diff=-1 best_diff: -inf -> -1
  i=1 s[i]='1' left='11'     zeros=0 ones=2 diff=-2 best_diff: -1 -> -1
  i=2 s[i]='1' left='111'    zeros=0 ones=3 diff=-3 best_diff: -1 -> -1
  ones_total = 4 -> đáp án = 4 + -1 = 3


3

Ví dụ `"1111"` cho thấy `best_diff` **âm** (−1) — phải khởi tạo `-inf` (hoặc giá trị tại điểm cắt đầu tiên), **không** được khởi tạo `0`. Nếu khởi tạo `0` sẽ ra `4`, sai.

## 9. Code nộp LeetCode

In [13]:
class Solution:
    def maxScore(self, s: str) -> int:
        zeros = ones = 0
        best = float("-inf")
        for ch in s[:-1]:                 # mọi điểm cắt: left kết thúc trước ký tự cuối
            if ch == "0":
                zeros += 1
            else:
                ones += 1
            best = max(best, zeros - ones)
        ones += s[-1] == "1"
        return ones + best

print([Solution().maxScore(s) for s, _ in examples])

[5, 5, 3]


`s[:-1]` = mọi ký tự **trừ ký tự cuối** — đúng là các ký tự có thể là "ký tự cuối của `left`".

### Bản prefix sum (dễ giải thích)

In [14]:
class SolutionPrefix:
    def maxScore(self, s: str) -> int:
        n = len(s)
        ones = [0] * (n + 1)
        for i, ch in enumerate(s):
            ones[i + 1] = ones[i] + (ch == "1")
        return max((i - ones[i]) + (ones[n] - ones[i]) for i in range(1, n))

print([SolutionPrefix().maxScore(s) for s, _ in examples])

[5, 5, 3]


### Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Baseline | cắt + `count` mỗi điểm | `O(n²)` | `O(n)` (chuỗi con) | dễ nghĩ |
| 2. Prefix sum | `ones[i]` tra `O(1)` | `O(n)` | `O(n)` | chuẩn |
| 3. Cập nhật `±1` | dời điểm cắt | `O(n)` (2 lượt) | `O(1)` | trực quan |
| 4. Một lượt | `ones_total + max(zeros − ones)` | `O(n)` (1 lượt) | `O(1)` | gọn nhất |

## 10. Vì sao đúng?

**Tập điểm cắt:** `left = s[:i]`, `right = s[i:]` khác rỗng ⇔ `1 <= i <= n − 1`. Mọi cách cắt hợp lệ tương ứng đúng một `i` ⇒ max trên `i` là đáp án.

**Prefix sum:** bất biến `ones[k]` = số `'1'` trong `s[:k]` (quy nạp: `ones[0] = 0`, thêm `s[k]` thì cộng 1 nếu là `'1'`). Từ đó `zeros_left = i − ones[i]`, `ones_right = ones[n] − ones[i]`.

**Một lượt:** `score(i) = zeros(i) + ones_total − ones(i)`. Vì `ones_total` là hằng, `argmax score = argmax (zeros − ones)` ⇒ `max score = ones_total + max_i (zeros(i) − ones(i))`. Bất biến vòng lặp: sau khi xử lý `s[0..i]`, `zeros`, `ones` là số 0/1 trong `s[:i+1]`, `best` = max của hiệu tại các điểm cắt `1..i+1`.

## 11. Time / Space

Gọi `n = len(s)`.

- Baseline: `n − 1` điểm cắt × `O(n)` (slicing + count) ⇒ `O(n²)`; bộ nhớ `O(n)` cho chuỗi con tạm.
- Prefix: `O(n)` thời gian, `O(n)` bộ nhớ.
- Một lượt: `O(n)` thời gian, `O(1)` bộ nhớ (trong code nộp, `s[:-1]` tạo bản sao `O(n)`; dùng `range(len(s) - 1)` nếu muốn tránh).

In [15]:
import random, time
random.seed(1)
big = "".join(random.choice("01") for _ in range(500))
for f in [max_score_bruteforce, max_score_prefix, max_score_delta, max_score_one_pass]:
    t0 = time.perf_counter()
    for _ in range(200):
        r = f(big)
    print(f"{f.__name__:<22} -> {r} ({1000 * (time.perf_counter() - t0):6.1f} ms / 200 lần)")

max_score_bruteforce   -> 263 (  66.3 ms / 200 lần)
max_score_prefix       -> 263 (  12.6 ms / 200 lần)
max_score_delta        -> 263 (   6.4 ms / 200 lần)
max_score_one_pass     -> 263 (   6.9 ms / 200 lần)


## 12. Bộ test

Mẫu đề; độ dài 2 (4 trường hợp `"00"`, `"01"`, `"10"`, `"11"`); toàn `0`; toàn `1`; dạng `"0...01...1"` (đạt điểm tối đa `n`); dạng `"1...10...0"`; và **đối chiếu** mọi cách với baseline trên **mọi** chuỗi nhị phân độ dài 2..12.

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<18} PASS -> {actual}")

sol = Solution()
check("ví dụ 1", sol.maxScore("011101"), 5)
check("ví dụ 2", sol.maxScore("00111"), 5)
check("ví dụ 3", sol.maxScore("1111"), 3)
check("'00'", sol.maxScore("00"), 1)
check("'01'", sol.maxScore("01"), 2)
check("'10'", sol.maxScore("10"), 0)
check("'11'", sol.maxScore("11"), 1)
check("toàn 0 (n=500)", sol.maxScore("0" * 500), 499)
check("0..01..1", sol.maxScore("0" * 7 + "1" * 5), 12)
check("1..10..0", sol.maxScore("1" * 3 + "0" * 3), 2)

ví dụ 1            PASS -> 5
ví dụ 2            PASS -> 5
ví dụ 3            PASS -> 3
'00'               PASS -> 1
'01'               PASS -> 2
'10'               PASS -> 0
'11'               PASS -> 1
toàn 0 (n=500)     PASS -> 499
0..01..1           PASS -> 12
1..10..0           PASS -> 2


In [17]:
from itertools import product

methods = [max_score_prefix, max_score_delta, max_score_one_pass, Solution().maxScore, SolutionPrefix().maxScore]
total = 0
for n in range(2, 13):
    for bits in product("01", repeat=n):
        s = "".join(bits)
        expected = max_score_bruteforce(s)
        for f in methods:
            assert f(s) == expected, (f, s)
        total += 1
print(f"Mọi cách khớp baseline trên toàn bộ {total} chuỗi nhị phân độ dài 2..12")

Mọi cách khớp baseline trên toàn bộ 8188 chuỗi nhị phân độ dài 2..12


`itertools.product("01", repeat=n)` sinh **mọi** bộ `n` ký tự lấy từ `"01"` — tức mọi chuỗi nhị phân độ dài `n` (`2^n` chuỗi). Kiểm thử **toàn bộ** miền nhỏ là cách mạnh để bắt lỗi biên.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Cho phép cắt tại `i = 0` hoặc `i = n` (một phần rỗng)

In [18]:
def wrong_range(s):
    return max(s[:i].count("0") + s[i:].count("1") for i in range(0, len(s) + 1))   # SAI

print("'1111': sai:", wrong_range("1111"), "| đúng:", max_score_bruteforce("1111"))
print("'0000': sai:", wrong_range("0000"), "| đúng:", max_score_bruteforce("0000"))

'1111': sai: 4 | đúng: 3
'0000': sai: 4 | đúng: 3


### Lỗi 2: Khởi tạo `best_diff = 0` trong cách một lượt

In [19]:
def wrong_init(s):
    zeros = ones = 0
    best_diff = 0                          # SAI: hiệu có thể âm ở mọi điểm cắt
    for i in range(len(s) - 1):
        zeros += s[i] == "0"
        ones += s[i] == "1"
        best_diff = max(best_diff, zeros - ones)
    return ones + (s[-1] == "1") + best_diff

print("'1111': sai:", wrong_init("1111"), "| đúng:", max_score_one_pass("1111"))

'1111': sai: 4 | đúng: 3


### Lỗi 3: Duyệt cả ký tự cuối trong vòng lặp cập nhật

Nếu vòng lặp chạy đủ `n` ký tự (`for ch in s`) thì xét cả điểm cắt `i = n` (right rỗng).

In [20]:
def wrong_full_loop(s):
    score, best = s.count("1"), 0
    for ch in s:                           # SAI: thừa 1 bước (right rỗng)
        score += 1 if ch == "0" else -1
        best = max(best, score)
    return best

print("'0000': sai:", wrong_full_loop("0000"), "| đúng:", max_score_delta("0000"))

'0000': sai: 4 | đúng: 3


### Lỗi 4: So sánh với số `0` thay vì ký tự `'0'`

`s[i] == 0` luôn `False` vì `s[i]` là **chuỗi**. Python không báo lỗi — chỉ âm thầm sai.

In [21]:
s = "0101"
print("s[0] == 0  :", s[0] == 0)
print("s[0] == '0':", s[0] == "0")

s[0] == 0  : False
s[0] == '0': True


## 14. Mẹo nhận dạng pattern

- "Chia mảng/chuỗi thành **trái | phải**, điểm phụ thuộc thống kê mỗi bên" ⇒ **prefix sum** (và/hoặc suffix sum), thử mọi điểm cắt trong `O(n)`.
- Dời điểm cắt 1 ô chỉ đổi phe 1 phần tử ⇒ cập nhật `O(1)`.
- Biểu thức có **hằng số** cộng thêm ⇒ tách hằng số ra ngoài `max` để đơn giản hóa.
- Luôn kiểm tra **biên của điểm cắt** (`1..n−1` khi hai phần phải khác rỗng).

## 15. Biến thể

1. Cho phép phần rỗng (`i = 0..n`).
2. Trả về **vị trí cắt** đạt điểm cao nhất (nhỏ nhất nếu hòa).
3. Điểm = số `'1'` bên trái + số `'0'` bên phải (đảo vai trò).
4. LeetCode 2483 *Minimum Penalty for a Shop* và LeetCode 926 *Flip String to Monotone Increasing* dùng cùng ý tưởng điểm cắt + prefix.

In [22]:
def best_split_index(s: str) -> int:
    best, best_i = float("-inf"), -1
    for i in range(1, len(s)):
        sc = s[:i].count("0") + s[i:].count("1")
        if sc > best:
            best, best_i = sc, i
    return best_i

def min_flips_mono_incr(s: str) -> int:          # LeetCode 926
    # cắt tại i: bên trái phải toàn 0 (lật các '1'), bên phải toàn 1 (lật các '0')
    ones_left, zeros_right = 0, s.count("0")
    best = zeros_right                            # i = 0
    for ch in s:
        if ch == "1":
            ones_left += 1
        else:
            zeros_right -= 1
        best = min(best, ones_left + zeros_right)
    return best

print(best_split_index("011101"), best_split_index("00111"))
print(min_flips_mono_incr("00110"), min_flips_mono_incr("010110"), min_flips_mono_incr("00011000"))

1 2
1 2 2


## 16. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** `s = "100101"`. Lập bảng điểm cho mọi điểm cắt hợp lệ, cho đáp án.

**Bài 2 (đoán output):** `"0110"[:1]`, `"0110"[1:]`, `"0110"[:-1]`, `3 + ("1" == "1")`, `"1" == 1`.

**Bài 3 (điền chỗ trống):** `exercise_max_score(s)` bằng prefix sum.

**Bài 4 (tự viết):** `exercise_max_score_one_pass(s)` — một lượt, `O(1)` bộ nhớ.

**Bài 5 (biến thể):** `exercise_max_score_reversed(s)` — điểm = số `'1'` bên trái + số `'0'` bên phải (hai phần khác rỗng).

**Bài 6 (mini challenge):** `exercise_max_score_three(s)` — cắt thành **3** phần khác rỗng `A | B | C`; điểm = số `'0'` trong `A` + số `'1'` trong `B` + số `'0'` trong `C`. Trả về điểm lớn nhất (`len(s) >= 3`). Yêu cầu `O(n²)` hoặc tốt hơn.

In [23]:
def exercise_max_score(s: str) -> int:
    # TODO 1: ones = [0] * (n + 1), xây prefix
    # TODO 2: duyệt i = 1..n-1
    # TODO 3: zeros_left + ones_right, giữ max
    pass

def exercise_max_score_one_pass(s: str) -> int:
    # TODO
    pass

# assert exercise_max_score("011101") == 5
# assert exercise_max_score_one_pass("1111") == 3

In [24]:
def exercise_max_score_reversed(s: str) -> int:
    # TODO
    pass

def exercise_max_score_three(s: str) -> int:
    # TODO
    pass

# assert exercise_max_score_reversed("011101") == 4
# assert exercise_max_score_three("01010") == 4

## 17. Gợi ý 3 mức

**Bài 1** — (1) Có 5 điểm cắt. (2) Tính `zeros_left` và `ones_right` cho từng `i`. (3) So sánh 5 giá trị.

**Bài 3** — (1) `ones[i + 1] = ones[i] + (ch == "1")`. (2) `zeros_left = i − ones[i]`. (3) `ones_right = ones[n] − ones[i]`.

**Bài 4** — (1) `zeros`, `ones`, `best = -inf`. (2) Duyệt `s[:-1]`, cập nhật rồi `best = max(best, zeros − ones)`. (3) Cộng ký tự cuối vào `ones`, trả `ones + best`.

**Bài 5** — (1) `score = ones_left + zeros_right`. (2) `= zeros_total + (ones_left − zeros_left)`. (3) Đổi vai trò `'0'`/`'1'` trong cách một lượt.

**Bài 6** — (1) Hai điểm cắt `1 <= i < j <= n − 1`. (2) Prefix `zeros[k]`, `ones[k]` cho phép tính mỗi phần `O(1)`. (3) Duyệt mọi cặp `(i, j)` ⇒ `O(n²)`; nâng cao: với mỗi `j` giữ max theo `i` để còn `O(n)`.

## 18. Đáp án tham khảo

**Bài 1:** `"100101"`: `i=1`: `"1"|"00101"` → 0 + 2 = 2; `i=2`: `"10"|"0101"` → 1 + 2 = 3; `i=3`: `"100"|"101"` → 2 + 2 = **4**; `i=4`: `"1001"|"01"` → 2 + 1 = 3; `i=5`: `"10010"|"1"` → 3 + 1 = **4**. Đáp án **4**.

**Bài 2:** `"0"`, `"110"`, `"011"`, `4`, `False`.

In [25]:
def answer_max_score(s):
    n = len(s)
    ones = [0] * (n + 1)
    for i, ch in enumerate(s):
        ones[i + 1] = ones[i] + (ch == "1")
    best = 0
    for i in range(1, n):
        best = max(best, (i - ones[i]) + (ones[n] - ones[i]))
    return best

def answer_max_score_one_pass(s):
    zeros = ones = 0
    best = float("-inf")
    for ch in s[:-1]:
        if ch == "0":
            zeros += 1
        else:
            ones += 1
        best = max(best, zeros - ones)
    return ones + (s[-1] == "1") + best

def answer_max_score_reversed(s):
    zeros = ones = 0
    best = float("-inf")
    for ch in s[:-1]:
        if ch == "0":
            zeros += 1
        else:
            ones += 1
        best = max(best, ones - zeros)
    return zeros + (s[-1] == "0") + best

def answer_max_score_three(s):
    n = len(s)
    z = [0] * (n + 1)                      # z[k] = số '0' trong s[:k]
    for k, ch in enumerate(s):
        z[k + 1] = z[k] + (ch == "0")
    best = 0
    for i in range(1, n - 1):              # A = s[:i]
        for j in range(i + 1, n):          # B = s[i:j], C = s[j:]
            a = z[i]
            b_ones = (j - i) - (z[j] - z[i])
            c = z[n] - z[j]
            best = max(best, a + b_ones + c)
    return best

def brute_three(s):
    return max(s[:i].count("0") + s[i:j].count("1") + s[j:].count("0")
               for i in range(1, len(s) - 1) for j in range(i + 1, len(s)))

assert answer_max_score("100101") == 4
assert ("0110"[:1], "0110"[1:], "0110"[:-1], 3 + ("1" == "1"), "1" == 1) == ("0", "110", "011", 4, False)
assert answer_max_score("011101") == 5 and answer_max_score_one_pass("1111") == 3
assert answer_max_score_reversed("011101") == 4
assert answer_max_score_three("01010") == 4
for n in range(2, 11):
    for bits in product("01", repeat=n):
        s = "".join(bits)
        assert answer_max_score(s) == answer_max_score_one_pass(s) == max_score_bruteforce(s)
        assert answer_max_score_reversed(s) == max(s[:i].count("1") + s[i:].count("0") for i in range(1, n))
        if n >= 3:
            assert answer_max_score_three(s) == brute_three(s)
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def maxScore(self, s: str) -> int:
        zeros = ones = 0
        best = float("-inf")
        for ch in s[:-1]:                  # điểm cắt 1..n-1
            if ch == "0":
                zeros += 1
            else:
                ones += 1
            best = max(best, zeros - ones) # score = ones_total + (zeros - ones)
        return ones + (s[-1] == "1") + best
```

**Khung tư duy "điểm cắt trái | phải":** viết điểm theo prefix → tách hằng số → một lượt duyệt lấy max.

**Checklist:**
- [ ] Điểm cắt chỉ từ `1` tới `n − 1` (hai phần khác rỗng).
- [ ] `best` khởi tạo `-inf` (hoặc giá trị điểm cắt đầu tiên), không phải `0` khi dùng hiệu.
- [ ] So sánh với ký tự `'0'`/`'1'`, không phải số.
- [ ] Nhớ cộng ký tự cuối vào tổng số `'1'`.